# EDA: Assignment 2 
- Big Data - 2025
- Juan Sebastian Vargas Torres

## Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
import matplotlib.cbook as cbook
import matplotlib.pyplot as plt

## Utils

In [0]:
def summarize_nulls(input_df: DataFrame) -> DataFrame:
    """
    Calculates the count and percentage of null values for each column in a Spark DataFrame.

    The function returns a new DataFrame with three columns:
    - column_name: The name of the column from the input DataFrame.
    - null_count: The total number of null values in that column.
    - null_percentage: The percentage of nulls relative to the total number of rows.

    Args:
        input_df: The Spark DataFrame to be analyzed.

    Returns:
        A new Spark DataFrame summarizing the null values, sorted by null_percentage descending.
    """
    # 1. Get the total number of rows to use for percentage calculation.
    total_rows = input_df.count()

    # If the DataFrame is empty, return an empty summary DataFrame with the correct schema.
    if total_rows == 0:
        schema = StructType([
            StructField("column_name", StringType(), False),
            StructField("null_count", LongType(), False),
            StructField("null_percentage", F.DoubleType(), False)
        ])
        return input_df.sparkSession.createDataFrame([], schema)

    # 2. Create an aggregation expression to count nulls for each column.
    null_counts_agg = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in input_df.columns]

    # 3. Execute the aggregation and collect the single result row into a dictionary.
    null_counts_dict = input_df.agg(*null_counts_agg).collect()[0].asDict()

    # 4. Transform the dictionary into a list of rows for the new summary DataFrame.
    summary_data = [(k, v) for k, v in null_counts_dict.items()]

    # Define the schema for our summary DataFrame
    summary_schema = StructType([
        StructField("column_name", StringType(), False),
        StructField("null_count", LongType(), False)
    ])

    # 5. Create the summary DataFrame.
    summary_df = input_df.sparkSession.createDataFrame(summary_data, schema=summary_schema)

    # 6. Add the null percentage column and sort for readability.
    summary_df = summary_df.withColumn(
        "null_percentage",
        F.floor(((F.col("null_count") / total_rows) * 100) * 100) / 100 # Truncate to 2 decimal places
    ).orderBy(F.desc("null_percentage"))

    return summary_df

## Reading Data

In [0]:
catalog = "workspace"
source_schema  = "bde"
source_volume  = "assignment2"

GREEN_DST = f"/Volumes/{catalog}/{source_schema}/{source_volume}/green"
YELLOW_DST = f"/Volumes/{catalog}/{source_schema}/{source_volume}/yellow"
LOCATION_DATA_PATH = f"{catalog}.{source_schema}.location"

In [0]:
green_df  = spark.read.parquet(GREEN_DST)
yellow_df = spark.read.parquet(YELLOW_DST)
location_df = spark.table(LOCATION_DATA_PATH)

## Exploring Data

### NYC Green Taxi Trip Records Data Dictionary

This table provides a detailed breakdown of the fields available in the NYC Green Taxi trip records dataset.

| Field Name | Description |
| --- | --- |
| **VendorID** | A code indicating the LPEP provider that provided the record. <br> 1 = Creative Mobile Technologies, LLC <br> 2 = Curb Mobility, LLC <br> 6 = Myle Technologies Inc |
| **lpep_pickup_datetime** | The date and time when the meter was engaged. |
| **lpep_dropoff_datetime** | The date and time when the meter was disengaged. |
| **store_and_fwd_flag** | This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka "store and forward," because the vehicle did not have a connection to the server. |
| **RatecodeID** | The final rate code in effect at the end of the trip. <br> 1 = Standard rate <br> 2 = JFK <br> 3 = Newark <br> 4 = Nassau or Westchester <br> 5 = Negotiated fare <br> 6 = Group ride <br> 99 = Null/unknown |
| **PULocationID** | TLC Taxi Zone in which the taximeter was engaged. |
| **DOLocationID** | TLC Taxi Zone in which the taximeter was disengaged. |
| **passenger_count** | The number of passengers in the vehicle. |
| **trip_distance** | The elapsed trip distance in miles reported by the taximeter. |
| **fare_amount** | The time-and-distance fare calculated by the meter. |
| **extra** | Miscellaneous extras and surcharges. |
| **mta_tax** | Tax that is automatically triggered based on the metered rate in use. |
| **tip_amount** | Tip amount - This field is automatically populated for credit card tips. Cash tips are not included. |
| **tolls_amount** | Total amount of all tolls paid in trip. |
| **improvement_surcharge** | Improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015. |
| **total_amount** | The total amount charged to passengers. Does not include cash tips. |
| **payment_type** | A numeric code signifying how the passenger paid for the trip. <br> 0 = Flex Fare trip <br> 1 = Credit card <br> 2 = Cash <br> 3 = No charge <br> 4 = Dispute <br> 5 = Unknown <br> 6 = Voided trip |
| **trip_type** | A code indicating whether the trip was a street-hail or a dispatch that is automatically assigned based on the metered rate in use but can be altered by the driver. <br> 1 = Street-hail <br> 2 = Dispatch |
| **congestion_surcharge** | Total amount collected in trip for NYS congestion surcharge. |
| **cbd_congestion_fee** | Per-trip charge for MTA's Congestion Relief Zone starting Jan. 5, 2025. |

In [0]:
display(green_df.limit(5))

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
2,2014-01-01T00:17:26.000Z,2014-01-01T00:37:11.000Z,N,1.0,17,225,1.0,2.28,13.5,0.5,0.5,0.0,0.0,null,null,14.5,2.0,null,null
1,2014-01-01T00:29:12.000Z,2014-01-01T00:37:43.000Z,N,1.0,127,241,1.0,2.1,9.0,0.0,0.5,0.0,0.0,null,null,9.5,2.0,null,null
2,2014-01-01T00:31:35.000Z,2014-01-01T00:44:09.000Z,N,1.0,166,243,1.0,4.72,15.5,0.5,0.5,4.0,0.0,null,null,20.5,1.0,null,null
2,2014-01-01T00:07:01.000Z,2014-01-01T00:21:54.000Z,N,1.0,7,157,1.0,2.88,13.0,0.5,0.5,2.88,0.0,null,null,16.88,1.0,null,null
2,2014-01-01T00:26:43.000Z,2014-01-01T00:37:17.000Z,N,1.0,83,197,2.0,3.8,13.0,0.5,0.5,0.0,0.0,null,null,14.0,2.0,null,null


In [0]:
green_df.count()

83484688

In [0]:
display(green_df.describe())

summary,VendorID,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
count,83484688,81599542,81599542,83484688,83484688,81599542,83484688,83484688,83484688,83484688,83484688,83484688,3052,69293600,83484688,81599542,79816787,9061528
mean,1.8039002553378412,null,1.1050638617554986,115.09322008845503,130.90132769017475,1.369136912557671,6.181435911097235,12.961640538569803,0.39768288862744466,0.4826802597621253,1.173449862806494,0.14810818170659273,0.024279161205766717,0.2959893248452346,15.488083903813193,1.517479277028295,1.0239124884844086,0.5117569354749
stddev,0.39711371847644295,null,0.7093115476170027,76.41039036527515,77.24924122866254,1.0475780249049664,609.9133426400977,11.484550078626839,0.5554099399403256,0.1127131711058026,2.6504039517458526,1.5117436807241567,0.2162642160160209,0.12707032589701697,109.70098052722514,0.5234415308048589,0.1527765743539236,1.07015583487471
min,1,N,1.0,1,1,0.0,-20329.08,-890.0,-25.42,-49.5,-101.0,-80.0,0.0,-1.0,-890.3,1.0,1.0,-2.75
max,6,Y,99.0,265,265,48.0,360068.14,10445.84,565.84,82.91,2017.73,7999.92,1.95,1.77,989970.39,5.0,2.0,2.75


In [0]:
display(green_df.printSchema())

root
 |-- VendorID: long (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- trip_type: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [0]:
display(summarize_nulls(green_df))

column_name,null_count,null_percentage
ehail_fee,83481636,99.99
congestion_surcharge,74423160,89.14
improvement_surcharge,14191088,16.99
trip_type,3667901,4.39
store_and_fwd_flag,1885146,2.25
RatecodeID,1885146,2.25
passenger_count,1885146,2.25
payment_type,1885146,2.25
VendorID,0,0.0
lpep_pickup_datetime,0,0.0


### NYC Yellow Taxi Trip Records Data Dictionary

This table provides a detailed breakdown of the fields available in the NYC Yellow Taxi trip records dataset.

| Field Name | Description |
| --- | --- |
| **VendorID** | A code indicating the TPEP provider that provided the record. <br> 1 = Creative Mobile Technologies, LLC <br> 2 = Curb Mobility, LLC <br> 6 = Myle Technologies Inc <br> 7 = Helix |
| **tpep_pickup_datetime** | The date and time when the meter was engaged. |
| **tpep_dropoff_datetime** | The date and time when the meter was disengaged. |
| **passenger_count** | The number of passengers in the vehicle. |
| **trip_distance** | The elapsed trip distance in miles reported by the taximeter. |
| **RatecodeID** | The final rate code in effect at the end of the trip. <br> 1 = Standard rate <br> 2 = JFK <br> 3 = Newark <br> 4 = Nassau or Westchester <br> 5 = Negotiated fare <br> 6 = Group ride <br> 99 = Null/unknown |
| **store_and_fwd_flag** | This flag indicates whether the trip record was held in vehicle memory before sending to the vendor. <br> Y = store and forward trip <br> N = not a store and forward trip |
| **PULocationID** | TLC Taxi Zone in which the taximeter was engaged. |
| **DOLocationID** | TLC Taxi Zone in which the taximeter was disengaged. |
| **payment_type** | A numeric code signifying how the passenger paid for the trip. <br> 0 = Flex Fare trip <br> 1 = Credit card <br> 2 = Cash <br> 3 = No charge <br> 4 = Dispute <br> 5 = Unknown <br> 6 = Voided trip |
| **fare_amount** | The time-and-distance fare calculated by the meter. |
| **extra** | Miscellaneous extras and surcharges. |
| **mta_tax** | Tax that is automatically triggered based on the metered rate in use. |
| **tip_amount** | Tip amount - This field is automatically populated for credit card tips. Cash tips are not included. |
| **tolls_amount** | Total amount of all tolls paid in trip. |
| **improvement_surcharge** | Improvement surcharge assessed at the flag drop. The improvement surcharge began being levied in 2015. |
| **total_amount** | The total amount charged to passengers. Does not include cash tips. |
| **congestion_surcharge** | Total amount collected in trip for NYS congestion surcharge. |
| **airport_fee** | For pick up only at LaGuardia and John F. Kennedy Airports. |
| **cbd_congestion_fee** | Per-trip charge for MTA's Congestion Relief Zone starting Jan. 5, 2025. |

In [0]:
display(yellow_df.limit(5))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
1,2015-01-01T00:11:33.000Z,2015-01-01T00:16:48.000Z,1.0,1.0,1.0,N,41,166,1,5.7,0.5,0.5,1.4,0.0,0.0,8.4,null,null
1,2015-01-01T00:18:24.000Z,2015-01-01T00:24:20.000Z,1.0,0.9,1.0,N,166,238,3,6.0,0.5,0.5,0.0,0.0,0.0,7.3,null,null
1,2015-01-01T00:26:19.000Z,2015-01-01T00:41:06.000Z,1.0,3.5,1.0,N,238,162,1,13.2,0.5,0.5,2.9,0.0,0.0,17.4,null,null
1,2015-01-01T00:45:26.000Z,2015-01-01T00:53:20.000Z,1.0,2.1,1.0,N,162,263,1,8.2,0.5,0.5,2.37,0.0,0.0,11.87,null,null
1,2015-01-01T00:59:21.000Z,2015-01-01T01:05:24.000Z,1.0,1.0,1.0,N,236,141,3,6.0,0.5,0.5,0.0,0.0,0.0,7.3,null,null


In [0]:
yellow_df.count()

907982776

In [0]:
display(yellow_df.describe())

summary,VendorID,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
count,907982776,898422396,907982776,898422396,848302158,907982776,907982776,907982776,907982776,907982776,907982776,907982776,907982776,854232273,907982776,244929776,66545741
mean,1.584716123513779,1.5999960045519612,6.948436307992512,1.1413988893927796,null,162.92768943119248,161.0412722135161,1.3243202908509797,13.452631983869502,0.5701828551316032,0.49575563875013456,2.1892765943970827,0.35398106859012946,0.32320546521206145,17.84067849912758,2.2433664538198084,0.0881363203995279
stddev,0.5031437426390818,1.2457277211118385,7537.823210574615,3.055189071286488,null,66.55195757891367,70.49642377271492,0.5230773999702153,4668.446908880636,16.630466663020027,16.593368864272332,4668.090822819877,1.7683711677071954,0.29738650827121976,401.70559459694107,0.8016096239996571,0.3229387455008491
min,1,0.0,-4.08401244E7,1.0,N,1,1,0,-1.33391414E8,-80.0,-0.55,-493.22,-140.63,-1.0,-2567.8,-2.5,-1.25
max,7,208.0,1.346190631E8,99.0,Y,265,265,5,998310.03,500000.8,500000.5,1.3339136353E8,5743.51,4000.3,1.0000015E7,4.5,1.25


In [0]:
display(yellow_df.printSchema())

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [0]:
display(summarize_nulls(yellow_df))

column_name,null_count,null_percentage
airport_fee,841437035,92.67
congestion_surcharge,663053000,73.02
store_and_fwd_flag,59680618,6.57
improvement_surcharge,53750503,5.91
passenger_count,9560380,1.05
RatecodeID,9560380,1.05
VendorID,0,0.0
tpep_pickup_datetime,0,0.0
tpep_dropoff_datetime,0,0.0
trip_distance,0,0.0


### Taxi location lookup data

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| **LocationID** | Integer | A unique number identifying each location zone. |
| **Borough** | String (Text) | The New York City borough where the zone is located (e.g., Queens, Bronx, Manhattan). |
| **Zone** | String (Text) | The name of the specific neighborhood or area (e.g., Jamaica Bay, Alphabet City). |
| **service_zone** | String (Text) | The category of taxi service for that zone (e.g., Boro Zone, Yellow Zone, EWR). |

In [0]:
display(location_df.limit(5))

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone


In [0]:
location_df.count()

265

In [0]:
display(location_df.describe())

summary,LocationID,Borough,Zone,service_zone
count,265,265,265,265
mean,133.0,null,null,null
stddev,76.643112323722,null,null,null
min,1,Bronx,Allerton/Pelham Gardens,Airports
max,265,Unknown,Yorkville West,Yellow Zone
